In [ ]:
import os
import glob
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from sklearn.manifold import TSNE

# LangChain
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Cell 2: Configuration & API Keys

load_dotenv(override=True)

OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set")

In [ ]:


# Groq is our primary LLM (free and fast)
openai_client = OpenAI()


# Settings
DB_NAME           = "gorithm_vector_db"
KNOWLEDGE_BASE    = Path("knowledgebase/")
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"   # Free HuggingFace model, runs locally
CHUNK_SIZE        = 1000
CHUNK_OVERLAP     = 200

print(f"\nKnowledge base path: {KNOWLEDGE_BASE.resolve()}")
print(f"Vector DB: {DB_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## Part A: Load & Chunk the Knowledge Base


In [ ]:
# Cell 3: Load documents from the knowledge base folders

documents = []

for folder in KNOWLEDGE_BASE.iterdir():
    if not folder.is_dir():
        continue
    doc_type = folder.name
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = Path(doc.metadata["source"]).name
    documents.extend(folder_docs)


print(f"\nTotal documents loaded: {len(documents)}")


In [ ]:
# Cell 4: Split into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

print(f"Documents split into {len(chunks)} chunks")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")


## Part B: Embed Chunks & Build Chroma Vector Store

We use **HuggingFace's `all-MiniLM-L6-v2`** model to convert each chunk into a 384-dimensional vector. This model:


In [ ]:
# Cell 5: Create embeddings and build Chroma vector store

print("Loading HuggingFace embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Embedding model ready!")

# Delete existing collection to start fresh
if Path(DB_NAME).exists():
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Cleared existing vector store'{DB_NAME}'")

# Build and persist the vector store
print("\nBuilding Chroma vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

collection = vectorstore._collection
count = collection.count()
sample_emb = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

print(f"\nVector store created:")
print(f"  - Total vectors stored: {count:,}")
print(f"  - Embedding dimensions: {len(sample_emb):,} (all-MiniLM-L6-v2)")
print(f"  - Persisted to: ./{DB_NAME}/")

## Part C: Visualize the Vector Space


In [ ]:
# Cell 6: Prepare data for visualization

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors   = np.array(result["embeddings"])
docs_text = result["documents"]
metadatas = result["metadatas"]
doc_types = [m["doc_type"] for m in metadatas]

# Color map by document type
COLOR_MAP = {"properties": "royalblue", "experiences": "mediumseagreen", "services": "tomato"}
colors = [COLOR_MAP.get(t, "gray") for t in doc_types]

print(f"Data ready for visualization: {len(vectors)} vectors")
from collections import Counter
print(f"Breakdown: {dict(Counter(doc_types))}")

In [ ]:
# Cell 7: 2D t-SNE Visualization

tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_2d = tsne_2d.fit_transform(vectors)

fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0],
    y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=10, color=colors, opacity=0.85, line=dict(width=1, color="white")),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_2d.update_layout(
    title="2D Goria Knowledge Base — Vector Clusters (t-SNE)",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=850, height=600,
    template="plotly_white"
)
fig_2d.show()

## RAG System Setup

In [ ]:
# RAG System Setup

# k=3 keeps context concise (avoids token limit on Groq free tier)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

SYSTEM_PROMPT = """\
You are Gorix, a warm and knowledgeable guest assistant for Goria Travels (goria.com/hotels), \
a curated short-stay booking platform based in Lagos, Nigeria.

Your job is to help guests find the perfect property, experience, or service.
You are enthusiastic, friendly, and always recommend booking via goria.com/hotels.
All prices are in Nigerian Naira (₦). Always present prices clearly.

Use the context below to answer the guest's question accurately and concisely.
If information is not in the context, say so and suggest the guest visits goria.com/hotels.

RELEVANT CONTEXT:
{context}
"""

def build_messages(message: str, history: list, system_prompt: str) -> list:
    """Build a clean OpenAI-compatible message list.

    Handles any Gradio history format (tuples OR message dicts).
    IMPORTANT: strips any extra fields (e.g. Gradio 5 adds 'metadata')
    since Groq and Gemini only accept 'role' + 'content'.
    Caps history to last 4 messages (2 full turns) to stay within token limits.
    """
    normalised = []
    for item in history:
        if isinstance(item, dict) and "role" in item and item.get("content"):
            normalised.append({"role": item["role"], "content": str(item["content"])})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            if u: normalised.append({"role": "user",      "content": str(u)})
            if b: normalised.append({"role": "assistant", "content": str(b)})
            
    normalised = normalised[-4:]  # keep last 2 turns only
    return [{"role": "system", "content": system_prompt}] + normalised + [{"role": "user", "content": message}]

def assistant_chat(message: str, history: list) -> str:
    # Step 1: Semantic retrieval from Chroma
    docs = retriever.invoke(message)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Step 2: Build messages (clean, no extra fields)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = build_messages(message, history, system_prompt)

    # Step 3: Use OpenAI API
    try:
        response = openai_client.chat.completions.create(
            model="gpt-5-nano",
            messages=messages,
        )
        return response.choices[0].message.content
    except Exception as err:
        print(f"[API error] {type(err).__name__}: {err}")
        return "Sorry, I'm having trouble right now. Please try again or visit goria.com/hotels."


print("Gorix is ready!")


## Gradio UI Setup

In [ ]:
# Cell 10: Launch Gorix on Gradio

goria_theme = gr.themes.Soft(
    primary_hue=gr.themes.Color(
        c50="#eef6ff", c100="#d9ebff", c200="#bcdaff", c300="#8ec4ff",
        c400="#59a7ff", c500="#1e90ff", c600="#1a7de6", c700="#1565bf",
        c800="#104d99", c900="#0c3a73", c950="#082752",
    ),
)

with gr.Blocks(theme=goria_theme, title="Gorix — Goria Travels AI Booking Assistant") as demo:
    gr.Markdown(
        """## Gorix is your Goria Travels Booking Assistant
Powered by RAG. Ask me about properties, experiences, and services at **goria.com/hotels**.

"""
    )
    chat = gr.ChatInterface(
        fn=assistant_chat,
        type="messages",
        examples=[
            "What is your most popular property?",
            "I need a romantic place in Lagos for 2 nights, budget ₦60,000",
            "Do you have properties for a family of 5?",
            "What outdoor experiences do you offer?",
            "Can you arrange an airport pickup?",
            "How much is a night at your most expensive hotel?"
        ],
        chatbot=gr.Chatbot(type="messages", height=450)
    )

demo.launch(inbrowser=True)